# Emotion Arithmetic and Blending

Just as word2vec showed that `king - man + woman = queen`, this notebook tests whether emotion directions support meaningful arithmetic. Starting from a neutral embedding, we blend multiple emotion directions and observe the classifier's response.

**Experiments:**
1. **Pure directions** — neutral + 1.0 * emotion_direction for each emotion
2. **Emotion blends** — combining two emotions (e.g., 0.5 * happy + 0.5 * sad)
3. **Interpolation paths** — smooth transition between two emotions through neutral
4. **Dominance tests** — which emotion wins in a blend?

In [ ]:
from __future__ import annotations
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def running_in_colab():
    try: import google.colab; return True
    except ImportError: return False
IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)

def run_command(cmd, cwd=None):
    print("+", " ".join(cmd)); subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

def ensure_packages():
    pkgs = {"yaml":"pyyaml","pandas":"pandas","numpy":"numpy","matplotlib":"matplotlib",
            "seaborn":"seaborn","torch":"torch","transformers":"transformers","sklearn":"scikit-learn","tqdm":"tqdm"}
    missing = sorted({p for m,p in pkgs.items() if importlib.util.find_spec(m) is None})
    if missing: run_command([sys.executable,"-m","pip","install","-q",*missing])
    else: print("Required packages available.")

def find_local_project_root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent, cwd/"FinalProject"]:
        c = c.resolve()
        if (c/"configs"/"wav2vec.yaml").exists() and (c/"src").exists(): return c
    raise FileNotFoundError("Could not find project root.")

def clone_clean(cr):
    if cr.exists(): shutil.rmtree(cr)
    run_command(["git","clone","--depth","1",REPO_URL,str(cr)]); return cr

def prepare_roots():
    rt=Path("/content")/REPO_NAME; cl=Path("/content")/f"{REPO_NAME}-clean"
    if rt.exists() and (rt/".git").exists():
        try: run_command(["git","-C",str(rt),"fetch","origin"])
        except: pass
        st=subprocess.run(["git","-C",str(rt),"status","--short"],text=True,capture_output=True,check=True).stdout.splitlines()
        ab=subprocess.run(["git","-C",str(rt),"rev-list","--left-right","--count","HEAD...origin/main"],text=True,capture_output=True,check=False)
        a,b=(0,0) if ab.returncode!=0 or not ab.stdout.strip() else map(int,ab.stdout.strip().split())
        if not [l for l in st if l.strip()] and a==0:
            if b>0:
                try: run_command(["git","-C",str(rt),"pull","--ff-only","origin","main"]); cr=rt
                except: cr=clone_clean(cl)
            else: cr=rt
        else: cr=clone_clean(cl)
        return rt,cr
    elif rt.exists(): return rt,clone_clean(cl)
    else: run_command(["git","clone","--depth","1",REPO_URL,str(rt)]); return rt,rt

ensure_packages()
if IS_COLAB:
    from google.colab import drive; drive.mount('/content/drive',force_remount=False)
    PROJECT_ROOT,CODE_ROOT=prepare_roots()
else:
    PROJECT_ROOT=find_local_project_root(); CODE_ROOT=PROJECT_ROOT

os.chdir(PROJECT_ROOT)
for r in [str(CODE_ROOT),str(PROJECT_ROOT)]:
    while r in sys.path: sys.path.remove(r)
sys.path.insert(0,str(CODE_ROOT))
if str(PROJECT_ROOT)!=str(CODE_ROOT): sys.path.insert(1,str(PROJECT_ROOT))
for n in [n for n in sys.modules if n=="src" or n.startswith("src.")]: del sys.modules[n]
print("Project root:",PROJECT_ROOT); print("Code root:",CODE_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_output_dir = local_analysis_dir / 'emotion_arithmetic'
local_output_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root else None
drive_output_dir = drive_analysis_dir / 'emotion_arithmetic' if drive_analysis_dir else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir
analysis_source_dir = local_analysis_dir
if not (analysis_source_dir / 'embedding_arrays.npz').exists() and drive_analysis_dir and (drive_analysis_dir / 'embedding_arrays.npz').exists():
    analysis_source_dir = drive_analysis_dir

from src.analysis.emotion_vectors import (
    build_direction_vectors, compute_class_centroids, load_embedding_artifacts, linear_classifier_probabilities,
)
from src.analysis.extract_embeddings import load_trained_checkpoint
from src.analysis.advanced_analysis import blend_emotions, evaluate_blends, interpolation_path

artifacts = load_embedding_artifacts(analysis_source_dir)
label_names = list(artifacts.summary['label_names'])
reference_label = 'neutral'
reference_idx = label_names.index(reference_label)
label_ids = artifacts.true_label_ids
split_array = artifacts.metadata['split'].to_numpy()
train_mask = split_array == 'train'

final_layer_idx = int(artifacts.layer_embeddings.shape[1] - 1)
embeddings = artifacts.layer_embeddings[:, final_layer_idx]

centroids = compute_class_centroids(embeddings[train_mask], label_ids[train_mask], len(label_names))
directions = build_direction_vectors(centroids, reference_idx)
neutral_centroid = centroids[reference_idx]

checkpoint = load_trained_checkpoint(checkpoint_dir)
cw = checkpoint.model.classifier.weight.detach().cpu().numpy()
cb = checkpoint.model.classifier.bias.detach().cpu().numpy()

print(f'Labels: {label_names}')
print(f'Neutral centroid shape: {neutral_centroid.shape}')

## 1. Pure Direction Sanity Check

First verify that `neutral + direction_i` produces the expected emotion.

In [ ]:
pure_configs = [
    {'name': f'neutral + 1.0*{name}', 'weights': {idx: 1.0}}
    for idx, name in enumerate(label_names) if idx != reference_idx
]
pure_configs.insert(0, {'name': 'neutral (baseline)', 'weights': {}})

pure_df = evaluate_blends(neutral_centroid, directions, cw, cb, label_names, reference_idx, pure_configs)
pure_df.to_csv(local_output_dir / 'pure_direction_blends.csv', index=False)
display(pure_df.round(4))

## 2. Emotion Blends

What happens when we combine two emotion directions? Do blends produce sensible intermediate states, or does one emotion dominate?

In [ ]:
from itertools import combinations

non_ref_emotions = [(idx, name) for idx, name in enumerate(label_names) if idx != reference_idx]

# All pairwise 50/50 blends
blend_configs = []
for (idx_a, name_a), (idx_b, name_b) in combinations(non_ref_emotions, 2):
    blend_configs.append({
        'name': f'0.5*{name_a} + 0.5*{name_b}',
        'weights': {idx_a: 0.5, idx_b: 0.5},
    })

# A few interesting 3-way blends
if len(non_ref_emotions) >= 3:
    blend_configs.append({
        'name': '0.33*happy + 0.33*sad + 0.33*angry',
        'weights': {label_names.index('happy'): 0.33, label_names.index('sad'): 0.33, label_names.index('angry'): 0.33},
    })

blend_df = evaluate_blends(neutral_centroid, directions, cw, cb, label_names, reference_idx, blend_configs)
blend_df.to_csv(local_output_dir / 'emotion_blends.csv', index=False)
display(blend_df.round(4))

# Visualize blend probabilities as stacked bars
prob_cols = [c for c in blend_df.columns if c.startswith('prob_')]
fig, ax = plt.subplots(figsize=(14, 6))
blend_df[prob_cols].plot(kind='barh', stacked=True, ax=ax, colormap='tab10')
ax.set_yticklabels(blend_df['blend_name'], fontsize=8)
ax.set_xlabel('Probability')
ax.set_title('Classifier Probability Distribution for Emotion Blends')
ax.legend(title='Emotion', bbox_to_anchor=(1.0, 1.0), fontsize=8)
plt.tight_layout()
plt.savefig(local_output_dir / 'emotion_blend_probabilities.png', dpi=220, bbox_inches='tight')
plt.show()

## 3. Interpolation Paths

Smoothly interpolate between pairs of emotions through neutral. If the latent space is well-organized, we should see smooth probability transitions.

In [ ]:
# Select a few interesting pairs
interp_pairs = [
    ('happy', 'sad'),
    ('angry', 'fearful'),
    ('happy', 'angry'),
    ('sad', 'fearful'),
]

all_interp_dfs = []
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (emo_a, emo_b) in zip(axes.flat, interp_pairs):
    idx_a = label_names.index(emo_a)
    idx_b = label_names.index(emo_b)
    interp_df = interpolation_path(
        neutral_centroid, directions[idx_a], directions[idx_b],
        cw, cb, label_names, steps=21,
    )
    interp_df['path'] = f'{emo_a} <-> {emo_b}'
    all_interp_dfs.append(interp_df)

    prob_cols = [c for c in interp_df.columns if c.startswith('prob_')]
    for col in prob_cols:
        emo_name = col.replace('prob_', '')
        ax.plot(interp_df['alpha'], interp_df[col], label=emo_name, linewidth=2)
    ax.set_xlabel(f'alpha ({emo_a} <-- 0 --> {emo_b})')
    ax.set_ylabel('Probability')
    ax.set_title(f'{emo_a} <-> {emo_b}')
    ax.legend(fontsize=7)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Emotion Interpolation Paths Through Neutral', y=1.02)
plt.tight_layout()
plt.savefig(local_output_dir / 'interpolation_paths.png', dpi=220)
plt.show()

full_interp_df = pd.concat(all_interp_dfs, ignore_index=True)
full_interp_df.to_csv(local_output_dir / 'interpolation_paths.csv', index=False)

## 4. Blend Weight Sensitivity

For a single pair (happy + angry), sweep the mixing ratio from 100% happy to 100% angry and visualize the transition.

In [ ]:
happy_idx = label_names.index('happy')
angry_idx = label_names.index('angry')
ratios = np.linspace(0, 1, 21)

ratio_rows = []
for r in ratios:
    w_happy = 1.0 - r
    w_angry = r
    blended = blend_emotions(neutral_centroid, directions, {happy_idx: w_happy, angry_idx: w_angry})
    probs = linear_classifier_probabilities(blended[None, :], cw, cb)[0]
    row = {'happy_weight': float(w_happy), 'angry_weight': float(w_angry)}
    for idx, name in enumerate(label_names):
        row[f'prob_{name}'] = float(probs[idx])
    ratio_rows.append(row)

ratio_df = pd.DataFrame(ratio_rows)
ratio_df.to_csv(local_output_dir / 'blend_weight_sweep.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
for name in label_names:
    ax.plot(ratio_df['angry_weight'], ratio_df[f'prob_{name}'], label=name, linewidth=2)
ax.set_xlabel('Angry Weight (Happy = 1 - Angry)')
ax.set_ylabel('Probability')
ax.set_title('Blend Weight Sweep: Happy <-> Angry')
ax.legend()
plt.tight_layout()
plt.savefig(local_output_dir / 'blend_weight_sweep.png', dpi=220)
plt.show()

In [ ]:
# --- Drive backup ---
if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping.')
elif drive_output_dir is None:
    print('Drive output directory is not configured.')
else:
    import shutil
    if drive_output_dir.exists(): shutil.rmtree(drive_output_dir)
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_output_dir)
    print('Backed up to:', drive_output_dir)

print('\nOutput files:')
for p in sorted(local_output_dir.iterdir()):
    print(f'  {p.name}')